# SynFS synthetic data example


In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve()
while not (PROJECT_ROOT / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

sys.path.insert(0, str(PROJECT_ROOT))


In [4]:


import numpy as np
import torch
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt

from src.trainer.trainer import SynFSTrainer
from src.data.dataset import SimpleDataset
from omegaconf import OmegaConf

from src.models.synfs_model import SynFSModel 

In [5]:
views_dims = [250,250]

In [6]:
cfg = OmegaConf.create({
    "model": {
        "views_dims": views_dims,
        "hidden_dims": [32,32],
        "output_dim":2,
        "batch_norm":True,
        "dropout":True,
        "activation": "relu",
        "learning_rate": 1e-3,
        "s_learning_rate": 1e-3,
        "weight_decay": 1e-4,
        "s_lam": 0.1,
        "ns_lam": 1.07,
        "ns_alpha": 0.25,
    },
    "nr_epochs": 90,
    "seed": 0,
    "device": "cpu"
})

In [8]:
# ================================
# Section B: Synthetic Dataset (Exact Match)
# ================================
from src.data.synfs_synthetic import (
    generate_multi_dataset,
    split_dataset,
)
from torch.utils.data import DataLoader

device = "cuda" if torch.cuda.is_available() else "cpu"

# ---- Generate EXACT SAME data as original notebook ----
views, y, (a_gt, s_gt) = generate_multi_dataset(
    n=20000,
    dims=views_dims,
    seed=0
)

# ---- Train/Val/Test split ----
(tr_X_set, tr_y), (va_X_set, va_y), (te_X_set, te_y) = split_dataset(views, y)

# ---- Wrap into PyTorch loaders ----
train_data = SimpleDataset(tr_X_set, tr_y, device)
val_data   = SimpleDataset(va_X_set, va_y, device)


train_loader = DataLoader(train_data, batch_size=250, drop_last=True, shuffle=True)
val_loader   = DataLoader(val_data, batch_size=250, drop_last=False, shuffle=False)

print("Train size:", len(train_data))
print("Val size:", len(val_data))


validate: (array([0., 1.]), array([ 9876, 10124]))
Train size: 12800
Val size: 3200


In [9]:
xs, y = train_data[0]
len(xs), xs[0].shape, xs[1].shape # first sample dimension (n of views), 1st view dim, 2nd view dim


(2, torch.Size([250]), torch.Size([250]))

In [10]:
len(tr_X_set), tr_X_set[0].shape, tr_X_set[1].shape

(2, (12800, 250), (12800, 250))

In [ ]:
from src.utils.feature_importance import get_important_features

ModuleNotFoundError: No module named 'utils'

In [ ]:
model = SynFSModel(cfg.model).to(cfg.device)
trainer = SynFSTrainer(cfg,model)


history = {"train_loss": [], "val_auroc": []}
trainer.set_X_mean_set(train_loader)

for epoch in range(cfg.nr_epochs):
    train_metrics = trainer.train_epoch(train_loader)
    val_metrics   = trainer.validate_epoch(val_loader)

    history["train_loss"].append(train_metrics["loss"])
    history["val_auroc"].append(val_metrics["auroc"])


    if epoch % 30 == 0:
        print("======Epoch", epoch, "Train Loss:", train_metrics['loss'], "Val AUROC:", val_metrics['auroc'], "======")
        
        get_important_features(
            model,
            which="synergistic",
            threshold=0.7,
            verbose=True
        )
        get_important_features(
            model,
            which="non_synergistic",
            threshold=0.7,
            verbose=True
        )
        print('\n')

======Epoch 0 Train Loss: 1.4763750176803738 Val AUROC: 0.5420112518053677 ======
[synergistic] threshold=0.7
  total features     : 500
  selected features  : 0
  indices: []
[non_synergistic] threshold=0.7
  total features     : 500
  selected features  : 0
  indices: []




KeyboardInterrupt: 

## Evaluation

### Feature Discovery

In [ ]:
ns_gt = [ai - syn for ai, syn in zip(a_gt, s_gt)]
ns_gt = np.where(np.concatenate(ns_gt))[0]
s_gt = np.where(np.concatenate(s_gt))[0]
ground_truth_features = [ns_gt, s_gt]

In [ ]:
ground_truth_features

[array([  2, 253]), array([  0, 251])]

In [ ]:
from metric.metric import strict_jaccard,standard_metrics,tpr_fdr

In [ ]:
# Get group similarity and group structure.
s = [gate.cpu().numpy() for gate in model.get_gates(model.s_model)]
n = [gate.cpu().numpy() for gate in model.get_gates(model.ns_model)]
n_predicted = np.where(np.concatenate(n)>0.7)[0]
s_predicted = np.where(np.concatenate(s)>0.7)[0]
predicted_features = [n_predicted, s_predicted]

# Get group similarity and group structure.
tpr, fdr = tpr_fdr(ground_truth_features, predicted_features)
j_index, ntrue, npredicted = strict_jaccard(ground_truth_features, predicted_features)

print("ground_truth_syn |", ground_truth_features[0], ",ground_truth_non-syn | ", ground_truth_features[1])
print("predicted_syn |", predicted_features[0], ",predicted_non-syn | ", predicted_features[1] )
print(
    "Jaccard Index: {:.3f}, True Positive Rate: {:.3f}%, False Discovery Rate: {:.3f}%".format(
        j_index, tpr, fdr
    )
)


ground_truth_syn | [  2 253] ,ground_truth_non-syn |  [  0 251]
predicted_syn | [  2 253] ,predicted_non-syn |  [  0 251]
Jaccard Index: 1.000, True Positive Rate: 100.000%, False Discovery Rate: 0.000%


### Predictive Performance

In [ ]:
data = SimpleDataset(te_X_set, te_y, device=device)
testloader = DataLoader(data, batch_size=len(te_y))
           
res = []
for x, y in testloader:
    logits = trainer.predict(x)
    res.append(logits.detach().cpu().numpy())
logits = np.concatenate(res)

auroc, auprc, accuracy, f1 = standard_metrics(tr_y, te_y, logits, verbose=True)

auroc | 0.822, auprc | 0.817, accuracy | 0.729, f1 | 0.713
